# Chapter 34: Debugging PyTorch

        **Part VI - Evaluation, Debugging & Seeing Inside** · Companion notebook for *PyTorch From Ground Up, Volume I: Foundations*

        This notebook follows the chapter's progression with fresh, CPU-friendly examples. Type, change, break, and repair the code; the goal is fluency, not passive reading.

        ## Learning objectives

        - Read errors from the final line upward
- Debug shape, dtype, device, and numerical failures
- Inspect gradients systematically


In [ ]:
import torch

torch.manual_seed(42)
print("PyTorch:", torch.__version__)
print("Default device: cpu")


## Turn shape errors into evidence

Print the contract before the failing layer. Catching the example lets the notebook continue while preserving the real message.


In [ ]:
import torch.nn as nn

layer = nn.Linear(5, 2)
wrong = torch.randn(4, 7)
print("input:", wrong.shape, "expected final width:", layer.in_features)
try:
    layer(wrong)
except RuntimeError as error:
    print("caught:", str(error).splitlines()[-1])
fixed = nn.Linear(7, 2)(wrong)
print("fixed output:", fixed.shape)


## Check shape, dtype, device

A compact diagnostic helper makes the important tensor facts impossible to overlook.


In [ ]:
def describe(name, tensor):
    print(f"{name:10s} shape={tuple(tensor.shape)} dtype={tensor.dtype} device={tensor.device} finite={torch.isfinite(tensor).all().item()}")

x = torch.randn(3, 4)
labels = torch.tensor([0, 1, 0])
describe("features", x)
describe("labels", labels)


## Find bad values and gradients

Check loss before backward, use anomaly detection only while locating a problem, and inspect gradient norms before the optimizer step.


In [ ]:
model = nn.Linear(4, 1)
prediction = model(x)
target = torch.randn(3, 1)
loss = nn.functional.mse_loss(prediction, target)
assert torch.isfinite(loss), "loss became non-finite"
model.zero_grad()
with torch.autograd.detect_anomaly(check_nan=True):
    loss.backward()
total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
print("loss:", loss.item(), "gradient norm before clipping:", total_norm.item())


## Practice

        Work through these without looking back first.

        1. Create and fix a dtype mismatch.
2. Trace shapes through a three-layer model.
3. Deliberately produce inf, catch it with isfinite, and repair the calculation.

        <details><summary>Study habit</summary>

        Predict every output shape before running a cell. When something fails, read the final line of the error and print the shape, dtype, and device of every tensor involved.

        </details>


In [ ]:
# Your practice space. Replace `pass` with your solution.
pass


## Recap

You completed Chapter 34's companion lab. Before moving on, explain the central idea aloud and recreate the smallest useful example from memory.

**Next:** return to the book for the full explanation, visual reasoning, watch-outs, and chapter exercises.
